# Welcome to Federated Analysis Workshop!
## Contents
1. [Introduction](#1.-Introduction)
2. [Workbench](#2.-Workbench)
3. [Incidence and prevalence](#3.-Incidence-and-Prevalence)
    1. [Cohort discovery](#3.1-Cohort-discovery)
    2. [Design and submit your analysis](#3.2-Design-and-submit-your-analysis)
    3. [Collect results](#3.3-Collect-your-results)
    4. [Visualise results](#3.4-Visualise-results)
    5. [Aggregating results](#3.5-Aggregating-results)

## 1. Introduction

This Jupyter notebook is a tutorial for running analysis through Five Safes TES (5S-TES) using python packages, e.g., 5S-TES Workbench, to form the Darwin EU executors and send tasks to a 5S-TES submission layer and collect your results.

By the end of this notebook, you will have:
1. Defined an analysis using a Darwin EU and formed the executors
2. Connected to 5S-TES with the workbench
3. Submitted it to the TES server as a TES message
4. Fetched the outputs of your analysis and visualised them.

This notebook has some "cells" which is a group of executable Python code. To run the code/cell, you can simply click on the cell and press the play button or pressing `Shift+Enter`.

Before each cell, we will provide a description of the task and the expected outputs. The full guide on how to run Workbench is available [here](https://docs.federated-analytics.ac.uk/tutorials/submit-to-5s-tes/workbench).

Start by loading the code supporting this notebook. Select the cell below and run it.

In [ ]:
import secrets
import shlex
import string
import warnings
from datetime import date

import matplotlib.pyplot as plt
import pandas as pd
from five_safes_tes_workbench.workbench import Workbench
from i_p_utils import (
    combine_incidence_results,
    load_5s_tes_incidence_result,
    plot_incidence_results,
    plot_jittered,
)
from omop_metadata_utils import DistributionCodesets
from summarised_result import SummarisedResult
from tes_omop_callers.define_cohorts import DefineConceptCohort
from tes_omop_callers.incidence_prevalence import IncidencePrevalence
from tes_omop_callers.patient_profiles import AddCharacteristics, SummariseCohort
from tes_omop_callers.shell_bundlers import CohortShellBundler
from ui import CodesetDisplay

alphabet = string.ascii_lowercase + string.digits
cohort_name = shlex.quote("".join(secrets.choice(alphabet) for _ in range(8)))

warnings.filterwarnings('ignore')
print("Imported libraries, keep going!")

## 2. Forming the executors

In TES, executors are the what we want to run on the TREs. They include `image` (or the "code engine") and `commands` (or the commands to run on the code engine).
For this workshop, the image have been pre-defined, so you only need to define the commands. The commands are quite hard to type correctly (syntax wise), so we have some functions to help you. These functions will form a `bundle` which includes a combination of `tasks`, and then turn this to the list of commands to run on the code engine.

By the upcoming exercises, we will form two analysis bundles, to do two types of analysis: Incidence Prevalance and Summary of Patient Profiles. Details about these analysis are explained on the presentation. 

### Exercise 1: Cohort Discovery
Before doing anslysis, we need to know which OMOP concept IDs are available and relevant to our analysis. In this tutorial, let's say we are interested in patients with diabetes.

Run cell 1 and search "diabetes". Get 2 concepts IDs on the left-hand side and replace "..." of `concept_set` in cell 2 with them, separeted by a comma.

In [ ]:
# Cell 1 
codesets = DistributionCodesets(
    {
        "Nottingham TRE 01": "metadata/1242/output.json",
        "BII A*Star Singapore": "metadata/Singapore/output.json",
    }
)

codeset_display = CodesetDisplay(codesets)
codeset_display.display()

After filling the (...), run cell 2 to form the task which defines the cohort we want to do analysis on. This task will be used in two analysis bundles.

In [ ]:
# Cell 2
cohort_task = DefineConceptCohort(
    cohort_name=cohort_name,
    concept_set={cohort_name: [201254,201826]},
)
print("Good job! A new cohort is defined correctly.")

To do the Incidence Prevalance analysis, we need a task to get the estimate indidence of the "diabetes" cohort (defined in cell 2). Cell 3 will do this, plus bundle the new task with the task in cell 2. 
In this tutorial, let's say we are interested in the incidence from 1/1/1984 to 1/1/2004. You can change these dates, if you wish, and run the cell (again).

In [ ]:
# Cell 3
incidence_task = IncidencePrevalence(
    outcome_cohort_name=cohort_name,
    denominator_cohort_name=f"{cohort_name}_denominator",
    denominator_cohort_date_range=(
        date(1984, 1, 1),
        date(2004, 1, 1),
    ),
    estimate_incidence_output_path="outputs/incidence.csv",
)

incidence_bundler = CohortShellBundler(
    image="ghcr.io/health-informatics-uon/omop-r-tools:sha-07f2d58",
    tasks=[cohort_task, incidence_task],
)

print("Bundled Incidence Prevalence analysis successfully!")

To do the Summary of Patient Profiles analysis, first we need a task to enrich the profile of patients, then we can summarise the profile. 

There are many characteristics we can choose to enrich, e.g., age, sex, prior observation, future observation, in observation, etc. And this tutorial has set the default values for these characteristics. You can change the values to see how the results change (new task needs to be formed and submitted to the TES server).

Run cell 4 to form the tasks to enrich the profile of patients, summarise the profile and bundle them together with the Cohort defining task in cell 2.


In [ ]:
# Cell 4
add_char_task = AddCharacteristics(
    cohort_name=cohort_name,
    index_date="cohort_end_date",
    add_age=True,
    age_group=[(0,17), (18,64), (65,150)],
    missing_month=6,
    missing_day=15,
    age_impose_month=True,
    age_impose_day=True,
    add_sex=True,
    add_prior_observation=True,
    add_future_observation=True,
    add_in_observation=True,
    in_observation_window=[(0,30)],
    complete_interval=True,
    use_demographics=True
)

summarise_task = SummariseCohort(
    cohort_table_name=cohort_name,
    variables=["age","sex","prior_observation","future_observation","in_observation"],
    estimates=["count","percentage","median","q25","q75"]
)

patient_profiles_bundler = CohortShellBundler(
    image="ghcr.io/health-informatics-uon/omop-r-tools:sha-07f2d58",
    tasks=[cohort_task, add_char_task, summarise_task]
)
print("Patient profiles bundler is formed correctly.")

## 5S-TES Workbench and Visualisation

After forming the executors, we can use the Workbench to finally form the full TES message and submit the tasks to the TES server. Workbench can then help us retrieve the results from the Submission layer's storage. We also prepared some utilities to visualise the results. Let's do it!

### Exercise 2: Validate the connection to the Submission layer

Before submitting the tasks to the TES server, we need to validate the connection to the Submission layer. This is done by running cell 5. Your task is filling in the your name (in the beginning of the cell) and the username and password (in the `validate` function). Afterwards, run the cell to validate the connection.
Note: `text` or `string` should be inside double quotes.

In [ ]:
# Cell 5
researcher_name = "Thien Nguyen"

if researcher_name == "" or researcher_name is None:
    raise ValueError("Please fill in your name in the beginning of the cell.")

wb = Workbench()
wb.validate(
    project="BIIFedADemo",
    tres=["Nottingham TRE 01", "BII-FA-TRE-DEMO"],
    tes_base_url="https://api.5s-tes.federated-research.com/",
    keycloak_url="https://drs-core-identity.azurewebsites.net/",
    client_id="Dare-Control-S3",
    client_secret="tBqx2MQnMI9oEG1QDKNpq3EKkPZMs7M6",
    # Enter the username, and password
    username="ohdsi_demo_user",
    password="ohdsi_demo_pass",
)

# Testing the connection by fetching the outputs of a known task
wb.fetch_outputs(task_id=3677)

print("\n\n\n----------------------------")
print("You're connecting just fine!")

### Incidence Prevalance Analysis submission

Run cell 6, which will form the full TES message for the Incidence Prevalance Analysis and submit the tasks to the TES server.

In [ ]:
# Cell 6
wb.build_tes.custom(
    name=f"Incidence Prevalance - {researcher_name}",
    description="Testing the incidence omop caller",
    executors=[incidence_bundler.render_executor()],
    outputs=[
        {
            "name": "Analysis Results Location",
            "description": "Analysis Results Location",
            "url": "s3://",
            "path": "/outputs",
            "type": "DIRECTORY",
        }
    ],
)

wb.submit()

After submitting the task, you can check the status of your task on [the submission layer](https://5s-tes.federated-research.com/Submission/GetAllSubmissions) or by running cell 7, which will check the task's status before fetching the outputs.

In [ ]:
# Cell 7
incidence_output = wb.fetch_outputs(task_id=3777)

### Incidence Prevalance Analysis visualisation

Run cell 8 to read the outputs into the notebook and wrangle them into the right shape for analysis. Also, plot the results.
Run cell 9 to plot the combined results and compare them with the individual TREs.

In [ ]:
# Cell 8

# Read the outputs into the notebook and wrangle them into the right shape for analysis
incidence_results = load_5s_tes_incidence_result(incidence_output)

# Plot the results
plot_incidence_results(incidence_results)

In [ ]:
# Cell 9
overall_incidence_result = combine_incidence_results(list(incidence_results.values()))

overall_incidence_result.plot_incidence()

plot, ax = plt.subplots(figsize=(12, 8))

plot_jittered(incidence_results["Nottingham TRE 01"], ax=ax)
plot_jittered(incidence_results["BII-FA-TRE-DEMO"], offset_weeks=-8, ax=ax)
overall_incidence_result.plot_incidence(axes=ax)

ax.legend(["NOTTS 1", "BII SIN", "Combined"])

plt.show()

### Patient Profiles Analysis submission

Run cell 10, which will form the full TES message for the Patient Profiles Analysis and submit the tasks to the TES server.

In [ ]:
# Cell 10
wb.build_tes.custom(
    name=f"Patient Profiles - {researcher_name}",
    description="Testing the patientprofiles omop caller",
    executors=[patient_profiles_bundler.render_executor()],
    outputs=[
        {
            "name": "Analysis Results Location",
            "description": "Analysis Results Location",
            "url": "s3://",
            "path": "/outputs",
            "type": "DIRECTORY",
        }
    ],
)
wb.submit()

After submitting the task, you can check the status of your task on [the submission layer](https://5s-tes.federated-research.com/Submission/GetAllSubmissions) or by running cell 11, which will check the task's status before fetching the outputs.

In [ ]:
# Cell 11
characteristics_results = wb.fetch_outputs()

Run cell 12 to see the summary of the patient profiles from our analysis.

In [ ]:
# Cell 12
characteristics_data = {
    k: SummarisedResult(pd.read_csv(v[0]))
    for k,v in characteristics_results.items()
}

def group_tres(results):
    dfs = []
    for tre, result in results.items():
        df = result.data
        df["TRE"] = tre
        dfs.append(df)
    all_tres = pd.concat(dfs)
    level_cols = ["TRE", "group_level", "strata_level", "variable_level"]
    all_tres = all_tres.loc[~(df.cdm_name.isna())]
    
    return all_tres.pivot(
        index=level_cols,
        columns=["variable_name","estimate_name"],
        values="estimate_value"
    ).T.style.highlight_null(props="color: transparent;")

    

group_tres(characteristics_data)

### What next?
By running this example, you have learned how to use 5S-TES with the Workbench, congratulations!

If you want to learn more, check out the [workbench documentation](https://docs.federated-analytics.ac.uk/tutorials/submit-to-5s-tes/workbench), [other notebooks](https://github.com/Health-Informatics-UoN/5s-TES-notebooks/), and [our packages](https://github.com/Health-Informatics-UoN/omop-r-toolkit/) using the Darwin-EU tools.